#$$
 \textbf{Projects in Machine Learning (ML) and Artificial Intelligence (AI)}
$$

 # $$
 \textbf{CSCI 6967}
 $$

 # $$
 \textbf{Homework 5}
 $$

#**Task 3**

##Part 1 (10 points): Implement the scaled dot-product attention as discussed in class (lecture 14) from scratch (use NumPy and pandas only, no deep learning libraries are allowed for this step).

In [ ]:
import numpy as np

def scaled_dot_product_attention(Q, K, V):
    """
    Compute the scaled dot-product attention.

    Args:
        Q: Query matrix of shape (batch_size, seq_length, d_k)
        K: Key matrix of shape (batch_size, seq_length, d_k)
        V: Value matrix of shape (batch_size, seq_length, d_v)

    Returns:
        output: The attention output of shape (batch_size, seq_length, d_v)
        attention_weights: The attention weights of shape (batch_size, seq_length, seq_length)
    """
    # Step 1: Compute the dot products between Q and K^T
    # and scale the scores by sqrt(d_k) for numerical stability.
    d_k = Q.shape[-1]
    scores = np.matmul(Q, K.transpose(0, 2, 1)) / np.sqrt(d_k)

    # Step 2: Apply softmax to obtain the attention weights.
    # For numerical stability, subtract the maximum value in each score vector.
    scores_max = np.max(scores, axis=-1, keepdims=True)
    exp_scores = np.exp(scores - scores_max)
    attention_weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)

    # Step 3: Compute the final output as a weighted sum of the values.
    output = np.matmul(attention_weights, V)

    return output, attention_weights

# --- Example Usage ---

# Create dummy data for a single batch with a sequence length of 3.
batch_size = 1
seq_length = 3
d_k = 4  # Dimension for queries and keys
d_v = 4  # Dimension for values

# Generate random Q, K, V matrices
np.random.seed(42)  # For reproducibility
Q = np.random.rand(batch_size, seq_length, d_k)
K = np.random.rand(batch_size, seq_length, d_k)
V = np.random.rand(batch_size, seq_length, d_v)

# Compute the attention output and weights.
output, attention_weights = scaled_dot_product_attention(Q, K, V)

# Display the results.
print("Queries (Q):\n", Q)
print("\nKeys (K):\n", K)
print("\nValues (V):\n", V)
print("\nAttention Weights:\n", attention_weights)
print("\nOutput (Attended Values):\n", output)

Queries (Q):
 [[[0.37454012 0.95071431 0.73199394 0.59865848]
  [0.15601864 0.15599452 0.05808361 0.86617615]
  [0.60111501 0.70807258 0.02058449 0.96990985]]]

Keys (K):
 [[[0.83244264 0.21233911 0.18182497 0.18340451]
  [0.30424224 0.52475643 0.43194502 0.29122914]
  [0.61185289 0.13949386 0.29214465 0.36636184]]]

Values (V):
 [[[0.45606998 0.78517596 0.19967378 0.51423444]
  [0.59241457 0.04645041 0.60754485 0.17052412]
  [0.06505159 0.94888554 0.96563203 0.80839735]]]

Attention Weights:
 [[[0.31164829 0.37066075 0.31769096]
  [0.32266579 0.33486977 0.34246444]
  [0.33283257 0.33507568 0.33209175]]]

Output (Attended Values):
 [[[0.38238456 0.56336845 0.59419359 0.48028741]
  [0.36781777 0.59386381 0.59857094 0.49987658]
  [0.37190176 0.5920136  0.59070988 0.49675455]]]


##Part 2 (10 points): Pick any encoder-decoder seq2seq model (as discussed in class) and integrate the scaled dot-product attention in the encoder architecture. You may come up with your own technique of integration or adopt one from literature. Hint: See Bahdanau or Luong attention paper presented in class (lecture 14).

In [ ]:
import numpy as np

# -------------------------
# 1. Scaled Dot-Product Attention (from Part 1)
# -------------------------
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Compute scaled dot-product attention.

    Args:
        Q: Query matrix of shape (batch_size, seq_length, d_k)
        K: Key matrix of shape (batch_size, seq_length, d_k)
        V: Value matrix of shape (batch_size, seq_length, d_v)
        mask: Optional mask for padding (batch_size, seq_length, seq_length)

    Returns:
        output: Attention-weighted values (batch_size, seq_length, d_v)
        attention_weights: Attention scores (batch_size, seq_length, seq_length)
    """
    d_k = Q.shape[-1]
    scores = np.matmul(Q, K.transpose(0, 2, 1)) / np.sqrt(d_k)  # (Q ⋅ K^T) / sqrt(d_k)

    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)  # Mask padding tokens

    attention_weights = np.exp(scores) / np.sum(np.exp(scores), axis=-1, keepdims=True)
    output = np.matmul(attention_weights, V)

    return output, attention_weights

# -------------------------
# 2. Encoder with Attention
# -------------------------
class Encoder:
    def __init__(self, input_dim, hidden_dim):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.Wx = np.random.randn(hidden_dim, input_dim)  # Input weight matrix
        self.Wh = np.random.randn(hidden_dim, hidden_dim)  # Hidden state weight matrix
        self.b = np.zeros((hidden_dim, 1))  # Bias

    def forward(self, inputs):
        """
        Encodes the input sequence.

        Args:
            inputs: (batch_size, seq_length, input_dim)

        Returns:
            hidden_states: All hidden states (batch_size, seq_length, hidden_dim)
        """
        batch_size, seq_length, _ = inputs.shape
        hidden_states = np.zeros((batch_size, seq_length, self.hidden_dim))

        h_t = np.zeros((batch_size, self.hidden_dim))  # Initial hidden state

        for t in range(seq_length):
            x_t = inputs[:, t, :]  # Get the input at time step t
            h_t = np.tanh(np.dot(x_t, self.Wx.T) + np.dot(h_t, self.Wh.T) + self.b.T)
            hidden_states[:, t, :] = h_t  # Store hidden state

        return hidden_states  # Output hidden states

# -------------------------
# 3. Decoder with Attention
# -------------------------
class Decoder:
    def __init__(self, output_dim, hidden_dim):
        self.output_dim = output_dim
        self.hidden_dim = hidden_dim
        self.Wo = np.random.randn(output_dim, hidden_dim)  # Output weight matrix
        self.bo = np.zeros((output_dim, 1))  # Bias

    def forward(self, encoder_outputs, decoder_input):
        """
        Computes decoder output with attention.

        Args:
            encoder_outputs: (batch_size, seq_length, hidden_dim)
            decoder_input: (batch_size, hidden_dim)  # Previous decoder hidden state

        Returns:
            output: Decoder output (batch_size, output_dim)
        """
        attention_output, attention_weights = scaled_dot_product_attention(
            decoder_input[:, np.newaxis, :],  # Q = decoder hidden state
            encoder_outputs,  # K = encoder hidden states
            encoder_outputs  # V = encoder hidden states
        )

        context_vector = np.mean(attention_output, axis=1)  # Aggregate context

        output = np.dot(context_vector, self.Wo.T) + self.bo.T  # Compute final output
        return output, attention_weights

# -------------------------
# 4 Running the Model with Example Data
# -------------------------
# Example sequence length and dimensions
batch_size = 1
seq_length = 5
input_dim = 8  # Size of input embedding
hidden_dim = 16  # Hidden state dimension
output_dim = 10  # Size of output vocabulary

# Generate random input data
np.random.seed(42)
encoder_inputs = np.random.rand(batch_size, seq_length, input_dim)

# Instantiate Encoder & Decoder
encoder = Encoder(input_dim, hidden_dim)
decoder = Decoder(output_dim, hidden_dim)

# Forward pass through Encoder
encoder_outputs = encoder.forward(encoder_inputs)  # (batch_size, seq_length, hidden_dim)

# Initialize decoder input (typically last hidden state of encoder)
decoder_input = np.mean(encoder_outputs, axis=1)  # (batch_size, hidden_dim)

# Forward pass through Decoder
decoder_output, attention_weights = decoder.forward(encoder_outputs, decoder_input)

# Print results
print("\nEncoder Outputs:\n", encoder_outputs)
print("\nAttention Weights:\n", attention_weights)
print("\nDecoder Output:\n", decoder_output)



Encoder Outputs:
 [[[-8.66571694e-01  6.17234557e-01 -5.94379808e-01 -7.55608441e-01
    9.97989559e-01 -9.35509671e-01  8.21785921e-01 -4.70398309e-01
   -6.52051180e-01  9.35792094e-01  9.94359780e-01 -4.58551326e-01
   -3.25795414e-01 -4.09469635e-01 -3.90950017e-01  5.49379342e-01]
  [-9.17882029e-01  8.22169505e-01 -9.00328703e-01 -9.13781877e-01
    9.99944949e-01 -9.95511911e-01 -9.99993593e-01  9.97780863e-01
   -9.00277053e-01  8.55907054e-01  9.83938638e-01 -5.74862121e-04
    5.12558698e-01  9.97129659e-01  9.99666802e-01  6.74585149e-01]
  [ 9.33752907e-01  6.94396615e-01 -9.99329788e-01  3.69944949e-01
    9.99699869e-01 -9.99813820e-01  7.38528491e-02  4.36936839e-01
   -9.97518402e-01  9.98748176e-01  7.63232794e-01 -8.09709864e-01
   -8.04028736e-01 -9.99989921e-01  9.99998173e-01  9.95613111e-01]
  [-9.72216696e-01  9.99998765e-01  7.21153620e-02  9.17775308e-01
    9.96387823e-01 -9.99999976e-01  7.36899208e-01  9.99989375e-01
   -6.29296370e-01  3.74287811e-01  9.99

##Part 3 (5 points): Pick any public dataset of your choice (use a small-scale dataset like a subset of the Tatoeba or Multi30k dataset) for machine translation task. Train your model from Part 2 for the machine translation task. Evaluate test set by reporting the BLEU Score.

link: https://opus.nlpl.eu/Tatoeba/el&en/v2023-04-12/Tatoeba#download

In [ ]:
from google.colab import files
uploaded = files.upload()


Saving el-en.el.txt to el-en.el.txt
Saving el-en.en.txt to el-en.en.txt


In [ ]:
with open('el-en.en.txt', 'r', encoding='utf-8') as f_en, \
     open('el-en.el.txt', 'r', encoding='utf-8') as f_el:
    english_lines = f_en.readlines()
    greek_lines = f_el.readlines()

In [ ]:
import math
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import nltk
from nltk.translate.bleu_score import sentence_bleu

######################################################
# 1. LOAD & PREPROCESS DATA
######################################################

def load_data(file_path, num_lines=None):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.read().strip().split('\n')
    if num_lines is not None:
        lines = lines[:num_lines]
    return lines

def tokenize(sentence):
    return sentence.lower().split()

def build_vocab(tokenized_sentences):
    vocab = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}
    idx = 4
    for sentence in tokenized_sentences:
        for token in sentence:
            if token not in vocab:
                vocab[token] = idx
                idx += 1
    return vocab

def sentence_to_indices(tokens, vocab, add_sos_eos=False):
    indices = []
    if add_sos_eos:
        indices.append(vocab["<sos>"])
    indices.extend([vocab.get(token, vocab["<unk>"]) for token in tokens])
    if add_sos_eos:
        indices.append(vocab["<eos>"])
    return indices

def pad_sequences(seq_list, pad_value):
    max_len = max(len(seq) for seq in seq_list)
    padded = [seq + [pad_value]*(max_len - len(seq)) for seq in seq_list]
    return padded

# ----------------------------------------------------------------
# Here we read 1,200 lines total, so we can do 1,000 for train,
# 200 for test. Adjust if your dataset is bigger or smaller.
# ----------------------------------------------------------------
greek_file = "el-en.el.txt"
english_file = "el-en.en.txt"

num_lines = 1200  # read first 1200 lines from each file
greek_sentences = load_data(greek_file, num_lines)
english_sentences = load_data(english_file, num_lines)

# Tokenize
greek_tokens = [tokenize(s) for s in greek_sentences]
english_tokens = [tokenize(s) for s in english_sentences]

# Build vocab
src_vocab = build_vocab(greek_tokens)
tgt_vocab = build_vocab(english_tokens)
inv_tgt_vocab = {idx: token for token, idx in tgt_vocab.items()}

# Convert to indices
src_sequences = [sentence_to_indices(s, src_vocab, add_sos_eos=False) for s in greek_tokens]
tgt_sequences = [sentence_to_indices(s, tgt_vocab, add_sos_eos=True) for s in english_tokens]

# Pad
src_padded = pad_sequences(src_sequences, pad_value=src_vocab["<pad>"])
tgt_padded = pad_sequences(tgt_sequences, pad_value=tgt_vocab["<pad>"])

# Convert to tensors
src_tensor = torch.LongTensor(src_padded)
tgt_tensor = torch.LongTensor(tgt_padded)

# ----------------------------------------------------------------
# Manually pick the first 1000 lines for training, next 200 for test.
# That way, you'll have exactly 200 test lines to evaluate.
# ----------------------------------------------------------------
train_src = src_tensor[:1000]
train_tgt = tgt_tensor[:1000]
test_src  = src_tensor[1000:1200]
test_tgt  = tgt_tensor[1000:1200]

######################################################
# 2. DATASET & DATALOADER
######################################################

class TranslationDataset(Dataset):
    def __init__(self, src_data, tgt_data):
        self.src_data = src_data
        self.tgt_data = tgt_data
    def __len__(self):
        return len(self.src_data)
    def __getitem__(self, idx):
        return self.src_data[idx], self.tgt_data[idx]

train_dataset = TranslationDataset(train_src, train_tgt)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

######################################################
# 3. MODEL DEFINITIONS
######################################################

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, num_layers, batch_first=True)

    def forward(self, src):
        embedded = self.embedding(src)  # (batch, src_len, emb_dim)
        outputs, (hidden, cell) = self.lstm(embedded)
        return outputs, hidden, cell

class ScaledDotProductAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.scale = math.sqrt(hidden_dim)
    def forward(self, query, key, value):
        scores = torch.bmm(query, key.transpose(1,2)) / self.scale  # (batch, 1, src_len)
        attn_weights = torch.softmax(scores, dim=-1)
        context = torch.bmm(attn_weights, value)  # (batch, 1, hidden_dim)
        return context, attn_weights

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim + hidden_dim, hidden_dim, num_layers, batch_first=True)
        self.attention = ScaledDotProductAttention(hidden_dim)
        self.fc_out = nn.Linear(hidden_dim*2, output_dim)
    def forward(self, input, hidden, cell, encoder_outputs):
        input = input.unsqueeze(1)  # (batch, 1)
        embedded = self.embedding(input)  # (batch, 1, emb_dim)

        # use last hidden layer as query
        query = hidden[-1].unsqueeze(1)  # (batch, 1, hidden_dim)
        context, attn_weights = self.attention(query, encoder_outputs, encoder_outputs)

        lstm_input = torch.cat((embedded, context), dim=2)
        output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))

        output = output.squeeze(1)   # (batch, hidden_dim)
        context = context.squeeze(1) # (batch, hidden_dim)

        prediction = self.fc_out(torch.cat((output, context), dim=1))
        return prediction, hidden, cell, attn_weights

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.size(0)
        trg_len = trg.size(1)
        vocab_size = self.decoder.embedding.num_embeddings

        outputs = torch.zeros(batch_size, trg_len, vocab_size).to(self.device)

        encoder_outputs, hidden, cell = self.encoder(src)

        # first input = <sos>
        input = trg[:,0]
        for t in range(1, trg_len):
            output, hidden, cell, attn = self.decoder(input, hidden, cell, encoder_outputs)
            outputs[:,t,:] = output
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[:,t] if teacher_force else top1
        return outputs

# Build model
INPUT_DIM = len(src_vocab)
OUTPUT_DIM = len(tgt_vocab)
EMB_DIM = 64
HIDDEN_DIM = 128

enc = Encoder(INPUT_DIM, EMB_DIM, HIDDEN_DIM)
dec = Decoder(OUTPUT_DIM, EMB_DIM, HIDDEN_DIM)
model = Seq2Seq(enc, dec, device).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=tgt_vocab["<pad>"])

######################################################
# 4. TRAIN
######################################################

model.train()
num_epochs = 50  # Increase if you want more training
for epoch in range(num_epochs):
    epoch_loss = 0
    for src_batch, tgt_batch in train_loader:
        src_batch = src_batch.to(device)
        tgt_batch = tgt_batch.to(device)

        optimizer.zero_grad()
        output = model(src_batch, tgt_batch)  # (batch, trg_len, OUTPUT_DIM)

        # exclude the first token (<sos>) for loss
        output_dim = output.shape[-1]
        output = output[:,1:,:].reshape(-1, output_dim)
        tgt = tgt_batch[:,1:].reshape(-1)

        loss = criterion(output, tgt)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss/len(train_loader):.4f}")

######################################################
# 5. TEST / EVALUATION (200 lines)
######################################################

def translate_sentence(model, sentence, src_vocab, tgt_vocab, inv_tgt_vocab, device, max_len=50):
    model.eval()
    tokens = sentence.lower().split()
    src_indices = [src_vocab.get(t, src_vocab["<unk>"]) for t in tokens]
    src_tensor = torch.LongTensor(src_indices).unsqueeze(0).to(device)

    with torch.no_grad():
        enc_out, hidden, cell = model.encoder(src_tensor)

    trg_indices = [tgt_vocab["<sos>"]]
    for _ in range(max_len):
        trg_tensor = torch.LongTensor([trg_indices[-1]]).to(device)
        with torch.no_grad():
            output, hidden, cell, attn = model.decoder(trg_tensor, hidden, cell, enc_out)
        pred_token = output.argmax(1).item()
        trg_indices.append(pred_token)
        if pred_token == tgt_vocab["<eos>"]:
            break

    translation = [inv_tgt_vocab.get(idx, "<unk>") for idx in trg_indices]
    return translation

model.eval()

bleu_scores = []
test_size = test_src.size(0)

for i in range(test_size):
    # Convert test_src[i] to a Greek sentence
    src_indices = test_src[i].tolist()
    # Trim padding
    src_indices = [x for x in src_indices if x != src_vocab["<pad>"]]
    src_tokens = [k for k,v in src_vocab.items() if v in src_indices]
    src_sentence = " ".join(src_tokens)

    # Reference (English)
    tgt_indices = test_tgt[i].tolist()
    ref_tokens = []
    for idx in tgt_indices:
        if idx not in (tgt_vocab["<pad>"], tgt_vocab["<sos>"], tgt_vocab["<eos>"]):
            # find token
            ref_tokens.append([k for k,v in tgt_vocab.items() if v == idx][0])

    # Model translation
    translation = translate_sentence(model, src_sentence, src_vocab, tgt_vocab, inv_tgt_vocab, device)
    # Remove special tokens
    translation = [t for t in translation if t not in ["<sos>", "<eos>", "<pad>"]]

    score = sentence_bleu([ref_tokens], translation)
    bleu_scores.append(score)

print(f"\nTest set size: {test_size}")
print("Average BLEU Score:", sum(bleu_scores)/len(bleu_scores))


Epoch 1/50, Loss: 6.0270
Epoch 2/50, Loss: 5.3295
Epoch 3/50, Loss: 5.1762
Epoch 4/50, Loss: 5.0623
Epoch 5/50, Loss: 4.9416
Epoch 6/50, Loss: 4.8023
Epoch 7/50, Loss: 4.6490
Epoch 8/50, Loss: 4.4797
Epoch 9/50, Loss: 4.2833
Epoch 10/50, Loss: 4.0706
Epoch 11/50, Loss: 3.8632
Epoch 12/50, Loss: 3.6421
Epoch 13/50, Loss: 3.4253
Epoch 14/50, Loss: 3.2187
Epoch 15/50, Loss: 2.9773
Epoch 16/50, Loss: 2.7747
Epoch 17/50, Loss: 2.5402
Epoch 18/50, Loss: 2.3251
Epoch 19/50, Loss: 2.1337
Epoch 20/50, Loss: 1.9296
Epoch 21/50, Loss: 1.7828
Epoch 22/50, Loss: 1.5718
Epoch 23/50, Loss: 1.4375
Epoch 24/50, Loss: 1.2846
Epoch 25/50, Loss: 1.1699
Epoch 26/50, Loss: 1.0486
Epoch 27/50, Loss: 0.9657
Epoch 28/50, Loss: 0.8637
Epoch 29/50, Loss: 0.8110
Epoch 30/50, Loss: 0.7220
Epoch 31/50, Loss: 0.6624
Epoch 32/50, Loss: 0.5579
Epoch 33/50, Loss: 0.5054
Epoch 34/50, Loss: 0.4591
Epoch 35/50, Loss: 0.4068
Epoch 36/50, Loss: 0.3534
Epoch 37/50, Loss: 0.3185
Epoch 38/50, Loss: 0.2914
Epoch 39/50, Loss: 0.

/usr/local/lib/python3.11/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.11/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.11/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_


Test set size: 200
Average BLEU Score: 6.106693348777351e-80


Despite training on 1,000 lines and testing on 200, the model’s outputs likely differ significantly from the reference translations, causing the BLEU score to be near zero. In short, a small dataset, a simple architecture, and minimal hyperparameter tuning often lead to poor alignment between predicted and reference sentences, resulting in extremely low BLEU. Larger training sets, more advanced architectures, or further optimization of parameters would typically be needed to improve translation quality.

##Part 4 (30 points): In this part you are required to implement a simplified Transformer model from scratch (using Python and NumPy/PyTorch/TensorFlow with minimal highlevel abstractions) and apply it to a machine translation task (e.g., English-to-French or English-to-German translation) using the same dataset from part 3.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import nltk
from nltk.translate.bleu_score import sentence_bleu

######################################################
# 1. DATA LOADING & PREPROCESS
######################################################

def load_data(file_path, num_lines=None):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.read().strip().split('\n')
    if num_lines is not None:
        lines = lines[:num_lines]
    return lines

def tokenize(sentence):
    return sentence.lower().split()

def build_vocab(tokenized_sentences):
    vocab = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}
    idx = 4
    for sent in tokenized_sentences:
        for tok in sent:
            if tok not in vocab:
                vocab[tok] = idx
                idx += 1
    return vocab

def sentence_to_indices(tokens, vocab, add_sos_eos=False):
    indices = []
    if add_sos_eos:
        indices.append(vocab["<sos>"])
    for t in tokens:
        indices.append(vocab.get(t, vocab["<unk>"]))
    if add_sos_eos:
        indices.append(vocab["<eos>"])
    return indices

def pad_sequences(seq_list, pad_value):
    max_len = max(len(seq) for seq in seq_list)
    return [seq + [pad_value]*(max_len - len(seq)) for seq in seq_list]

# IMPORTANT: Swap the file assignments to use English as the source
src_file = "el-en.en.txt"   # English (source)
tgt_file = "el-en.el.txt"   # Greek (target)

# Use a larger dataset – here we load 1200 lines (e.g., 1000 for training, 200 for testing)
num_lines = 1200

src_lines = load_data(src_file, num_lines)
tgt_lines = load_data(tgt_file, num_lines)

# Tokenize
src_tokens = [tokenize(line) for line in src_lines]
tgt_tokens = [tokenize(line) for line in tgt_lines]

# Build vocabularies
src_vocab = build_vocab(src_tokens)
tgt_vocab = build_vocab(tgt_tokens)
inv_tgt_vocab = {v: k for k, v in tgt_vocab.items()}

# Convert sentences to indices
src_indices = [sentence_to_indices(toks, src_vocab, add_sos_eos=False) for toks in src_tokens]
tgt_indices = [sentence_to_indices(toks, tgt_vocab, add_sos_eos=True) for toks in tgt_tokens]

# Pad sequences
src_padded = pad_sequences(src_indices, pad_value=src_vocab["<pad>"])
tgt_padded = pad_sequences(tgt_indices, pad_value=tgt_vocab["<pad>"])

# Convert to torch tensors
src_tensor = torch.LongTensor(src_padded)
tgt_tensor = torch.LongTensor(tgt_padded)

# Split into training and testing sets: first 1000 for train, next 200 for test
train_src = src_tensor[:1000]
train_tgt = tgt_tensor[:1000]
test_src  = src_tensor[1000:]
test_tgt  = tgt_tensor[1000:]

class TranslationDataset(Dataset):
    def __init__(self, src, tgt):
        self.src = src
        self.tgt = tgt
    def __len__(self):
        return len(self.src)
    def __getitem__(self, idx):
        return self.src[idx], self.tgt[idx]

train_dataset = TranslationDataset(train_src, train_tgt)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

######################################################
# 2. POSITIONAL ENCODING
######################################################

class PositionalEncoding(nn.Module):
    def __init__(self, emb_dim, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, emb_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, emb_dim, 2).float() * (-math.log(10000.0) / emb_dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, emb_dim)
        self.register_buffer('pe', pe)

    def forward(self, x):
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len, :].to(x.device)
        return x

######################################################
# 3. TRANSFORMER SUB-COMPONENTS
######################################################

def subsequent_mask(size):
    mask = torch.triu(torch.ones(size, size), diagonal=1).bool()
    return mask.unsqueeze(0)  # (1, size, size)

class MultiHeadAttention(nn.Module):
    def __init__(self, emb_dim=64, num_heads=2):
        super().__init__()
        self.emb_dim = emb_dim
        self.num_heads = num_heads
        self.head_dim = emb_dim // num_heads

        self.w_q = nn.Linear(emb_dim, emb_dim)
        self.w_k = nn.Linear(emb_dim, emb_dim)
        self.w_v = nn.Linear(emb_dim, emb_dim)
        self.w_o = nn.Linear(emb_dim, emb_dim)

    def forward(self, query, key, value, mask=None):
        B, Q_len, _ = query.size()
        B, K_len, _ = key.size()

        Q = self.w_q(query)
        K = self.w_k(key)
        V = self.w_v(value)

        Q = Q.view(B, Q_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, K_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, K_len, self.num_heads, self.head_dim).transpose(1, 2)

        d_k = self.head_dim
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
        if mask is not None:
            if mask.dim() == 3:
                mask = mask.unsqueeze(1)
            scores = scores.masked_fill(mask, float('-inf'))
        attn = torch.softmax(scores, dim=-1)
        context = torch.matmul(attn, V)
        context = context.transpose(1, 2).contiguous().view(B, Q_len, self.emb_dim)
        out = self.w_o(context)
        return out, attn

class FeedForward(nn.Module):
    def __init__(self, emb_dim=64, ff_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, emb_dim)
        )
    def forward(self, x):
        return self.net(x)

class EncoderLayer(nn.Module):
    def __init__(self, emb_dim=64, num_heads=2, ff_dim=128):
        super().__init__()
        self.mha = MultiHeadAttention(emb_dim, num_heads)
        self.ffn = FeedForward(emb_dim, ff_dim)
        self.norm1 = nn.LayerNorm(emb_dim)
        self.norm2 = nn.LayerNorm(emb_dim)

    def forward(self, x, src_mask=None):
        attn_out, _ = self.mha(x, x, x, mask=src_mask)
        x = self.norm1(x + attn_out)
        ff_out = self.ffn(x)
        x = self.norm2(x + ff_out)
        return x

class DecoderLayer(nn.Module):
    def __init__(self, emb_dim=64, num_heads=2, ff_dim=128):
        super().__init__()
        self.mha_self = MultiHeadAttention(emb_dim, num_heads)
        self.mha_encdec = MultiHeadAttention(emb_dim, num_heads)
        self.ffn = FeedForward(emb_dim, ff_dim)
        self.norm1 = nn.LayerNorm(emb_dim)
        self.norm2 = nn.LayerNorm(emb_dim)
        self.norm3 = nn.LayerNorm(emb_dim)

    def forward(self, x, enc_out, tgt_mask=None, src_mask=None):
        _x, _ = self.mha_self(x, x, x, mask=tgt_mask)
        x = self.norm1(x + _x)
        _x, attn = self.mha_encdec(x, enc_out, enc_out, mask=src_mask)
        x = self.norm2(x + _x)
        ff_out = self.ffn(x)
        x = self.norm3(x + ff_out)
        return x, attn

class TransformerEncoder(nn.Module):
    def __init__(self, num_layers=2, emb_dim=64, num_heads=2, ff_dim=128, vocab_size=1000):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.pos_enc = PositionalEncoding(emb_dim)
        self.layers = nn.ModuleList([EncoderLayer(emb_dim, num_heads, ff_dim) for _ in range(num_layers)])
    def forward(self, src, src_mask=None):
        x = self.emb(src)
        x = self.pos_enc(x)
        for layer in self.layers:
            x = layer(x, src_mask=src_mask)
        return x

class TransformerDecoder(nn.Module):
    def __init__(self, num_layers=2, emb_dim=64, num_heads=2, ff_dim=128, vocab_size=1000):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.pos_enc = PositionalEncoding(emb_dim)
        self.layers = nn.ModuleList([DecoderLayer(emb_dim, num_heads, ff_dim) for _ in range(num_layers)])
    def forward(self, tgt, enc_out, tgt_mask=None, src_mask=None):
        x = self.emb(tgt)
        x = self.pos_enc(x)
        attn = None
        for layer in self.layers:
            x, attn = layer(x, enc_out, tgt_mask=tgt_mask, src_mask=src_mask)
        return x, attn

class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, emb_dim=64, num_heads=2, ff_dim=128, num_layers=2):
        super().__init__()
        self.encoder = TransformerEncoder(num_layers, emb_dim, num_heads, ff_dim, src_vocab_size)
        self.decoder = TransformerDecoder(num_layers, emb_dim, num_heads, ff_dim, tgt_vocab_size)
        self.fc_out = nn.Linear(emb_dim, tgt_vocab_size)

    def make_src_mask(self, src):
        return None

    def make_tgt_mask(self, tgt):
        batch_size, tgt_len = tgt.size()
        sub_mask = subsequent_mask(tgt_len).to(tgt.device)
        return sub_mask

    def forward(self, src, tgt):
        src_mask = self.make_src_mask(src)
        tgt_mask = self.make_tgt_mask(tgt)
        enc_out = self.encoder(src, src_mask=src_mask)
        dec_out, attn = self.decoder(tgt, enc_out, tgt_mask=tgt_mask, src_mask=src_mask)
        logits = self.fc_out(dec_out)
        return logits, attn

######################################################
# 4. TRAINING SETUP
######################################################

SRC_VOCAB_SIZE = len(src_vocab)
TGT_VOCAB_SIZE = len(tgt_vocab)

model = Transformer(
    src_vocab_size=SRC_VOCAB_SIZE,
    tgt_vocab_size=TGT_VOCAB_SIZE,
    emb_dim=64,
    num_heads=2,
    ff_dim=128,
    num_layers=2
).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=tgt_vocab["<pad>"])

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    epoch_loss = 0
    for src_batch, tgt_batch in loader:
        src_batch = src_batch.to(device)
        tgt_batch = tgt_batch.to(device)

        tgt_input = tgt_batch[:, :-1]
        tgt_labels = tgt_batch[:, 1:].contiguous().view(-1)

        optimizer.zero_grad()
        logits, _ = model(src_batch, tgt_input)
        logits = logits.view(-1, TGT_VOCAB_SIZE)
        loss = criterion(logits, tgt_labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
    return epoch_loss / len(loader)

######################################################
# 5. TRAIN LOOP
######################################################

num_epochs = 40
for epoch in range(num_epochs):
    loss = train_epoch(model, train_loader, optimizer, criterion)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss:.4f}")

######################################################
# 6. INFERENCE (GREEDY)
######################################################

def greedy_decode(model, src_sentence, max_len=50):
    model.eval()
    tokens = src_sentence.lower().split()
    src_indices = [src_vocab.get(t, src_vocab["<unk>"]) for t in tokens]
    src_tensor = torch.LongTensor(src_indices).unsqueeze(0).to(device)

    with torch.no_grad():
        enc_out = model.encoder(src_tensor)

    generated = [tgt_vocab["<sos>"]]
    for _ in range(max_len):
        tgt_input = torch.LongTensor(generated).unsqueeze(0).to(device)
        tgt_mask = model.make_tgt_mask(tgt_input)
        with torch.no_grad():
            dec_out, attn = model.decoder(tgt_input, enc_out, tgt_mask=tgt_mask)
            logits = model.fc_out(dec_out)
        next_token = logits[0, -1].argmax(-1).item()
        generated.append(next_token)
        if next_token == tgt_vocab["<eos>"]:
            break
    tokens_out = [inv_tgt_vocab.get(i, "<unk>") for i in generated]
    return tokens_out

######################################################
# 7. EVALUATE BLEU ON TEST SET
######################################################

model.eval()
bleu_scores = []
test_size = test_src.size(0)

for i in range(test_size):
    src_i = test_src[i].tolist()
    src_i = [x for x in src_i if x != src_vocab["<pad>"]]
    src_tokens = [k for k, v in src_vocab.items() if v in src_i]
    src_sentence = " ".join(src_tokens)

    tgt_i = test_tgt[i].tolist()
    ref_tokens = []
    for idx in tgt_i:
        if idx not in (tgt_vocab["<pad>"], tgt_vocab["<sos>"], tgt_vocab["<eos>"]):
            ref_tokens.append([k for k, v in tgt_vocab.items() if v == idx][0])

    pred = greedy_decode(model, src_sentence, max_len=50)
    pred = [t for t in pred if t not in ["<sos>", "<eos>", "<pad>"]]
    bleu = sentence_bleu([ref_tokens], pred)
    bleu_scores.append(bleu)

avg_bleu = sum(bleu_scores) / len(bleu_scores) if bleu_scores else 0.0
print("\nTest set size:", test_size)
print("Average BLEU Score:", avg_bleu)


Epoch 1/40, Loss: 6.5235
Epoch 2/40, Loss: 5.7901
Epoch 3/40, Loss: 5.4303
Epoch 4/40, Loss: 5.0643
Epoch 5/40, Loss: 4.6721
Epoch 6/40, Loss: 4.2742
Epoch 7/40, Loss: 3.8429
Epoch 8/40, Loss: 3.3929
Epoch 9/40, Loss: 2.9405
Epoch 10/40, Loss: 2.4932
Epoch 11/40, Loss: 2.0691
Epoch 12/40, Loss: 1.6636
Epoch 13/40, Loss: 1.2993
Epoch 14/40, Loss: 0.9820
Epoch 15/40, Loss: 0.7185
Epoch 16/40, Loss: 0.5218
Epoch 17/40, Loss: 0.3776
Epoch 18/40, Loss: 0.2798
Epoch 19/40, Loss: 0.2161
Epoch 20/40, Loss: 0.1790
Epoch 21/40, Loss: 0.1594
Epoch 22/40, Loss: 0.1313
Epoch 23/40, Loss: 0.1205
Epoch 24/40, Loss: 0.1059
Epoch 25/40, Loss: 0.1084
Epoch 26/40, Loss: 0.0985
Epoch 27/40, Loss: 0.0924
Epoch 28/40, Loss: 0.0879
Epoch 29/40, Loss: 0.0825
Epoch 30/40, Loss: 0.0815
Epoch 31/40, Loss: 0.0802
Epoch 32/40, Loss: 0.0820
Epoch 33/40, Loss: 0.0834
Epoch 34/40, Loss: 0.0753
Epoch 35/40, Loss: 0.0768
Epoch 36/40, Loss: 0.0761
Epoch 37/40, Loss: 0.0780
Epoch 38/40, Loss: 0.0740
Epoch 39/40, Loss: 0.

Despite employing a simplified Transformer (with only two encoder/decoder layers, two attention heads, and smaller embedding/feedforward sizes) and training on a limited dataset (1,000 lines for training, 200 for testing), the model’s BLEU score remains near zero. This outcome is expected given the small dataset size and the reduced model capacity, both of which make it difficult for the Transformer to learn robust alignments and produce accurate translations. For better performance, one would typically need more training data, additional training epochs, or advanced techniques (e.g., improved tokenization or hyperparameter tuning) to help the simplified Transformer model capture more complex linguistic patterns and achieve higher BLEU scores.